## এমবেডিংস

আমাদের আগের উদাহরণে, আমরা উচ্চ-মাত্রিক ব্যাগ-অফ-ওয়ার্ডস ভেক্টরগুলির মাধ্যমে কাজ করেছি যার দৈর্ঘ্য ছিল `vocab_size`, এবং আমরা স্পষ্টভাবে নিম্ন-মাত্রিক পজিশনাল রেপ্রেজেন্টেশন ভেক্টর থেকে স্পার্স ওয়ান-হট রেপ্রেজেন্টেশনে রূপান্তর করছিলাম। এই ওয়ান-হট রেপ্রেজেন্টেশন স্মৃতিসমর্থনকারী নয়, অতিরিক্তভাবে, প্রতিটি শব্দ একে অপরের থেকে স্বাধীনভাবে বিবেচনা করা হয়, অর্থাৎ ওয়ান-হট এনকোডেড ভেক্টরগুলি শব্দগুলির মধ্যে কোনো অর্থসম্পর্ক প্রদর্শন করে না।

এই ইউনিটে, আমরা **News AG** ডেটাসেটটি আরও অনুসন্ধান করব। শুরু করতে, ডেটা লোড করি এবং আগের নোটবুক থেকে কিছু সংজ্ঞা নিয়ে আসি।


In [1]:
import torch
import torchtext
import numpy as np
from torchnlp import *
train_dataset, test_dataset, classes, vocab = load_dataset()
vocab_size = len(vocab)
print("Vocab size = ",vocab_size)

Loading dataset...


d:\WORK\ai-for-beginners\5-NLP\14-Embeddings\data\train.csv: 29.5MB [00:01, 18.8MB/s]                            
d:\WORK\ai-for-beginners\5-NLP\14-Embeddings\data\test.csv: 1.86MB [00:00, 11.2MB/s]                          


Building vocab...
Vocab size =  95812



## এম্বেডিং কী?

**এম্বেডিং** এর ধারণা হল শব্দগুলোকে নিম্ন-মাত্রার ঘন ভেক্টর দ্বারা উপস্থাপন করা, যা কোনোভাবে শব্দের অর্থবোধক অর্থ প্রতিফলিত করে। আমরা পরে কীভাবে অর্থবোধক শব্দ এম্বেডিং তৈরি করতে হয় তা আলোচনা করব, কিন্তু এপর্যন্ত এম্বেডিংকে শুধুমাত্র একটি উপায় হিসেবে ভাবা যাক শব্দ ভেক্টরের মাত্রা কমানোর জন্য। 

তাই, এম্বেডিং লেয়ার একটি শব্দ ইনপুট হিসেবে নেবে, এবং নির্দিষ্ট `embedding_size` মাত্রার একটি আউটপুট ভেক্টর তৈরি করবে। একভাবে, এটি `Linear` লেয়ারের মতোই, কিন্তু এক-হট এনকোডেড ভেক্টর নেওয়ার বদলে, এটি শব্দের নম্বর ইনপুট হিসেবে নিতে পারবে।

আমাদের নেটওয়ার্কে প্রথম লেয়ার হিসেবে এম্বেডিং লেয়ার ব্যবহার করে, আমরা ব্যাগ-অফ-ওয়ার্ডস থেকে **এম্বেডিং ব্যাগ** মডেলে স্যুইচ করতে পারি, যেখানে প্রথমে আমাদের টেক্সটের প্রতিটি শব্দকে সংশ্লিষ্ট এম্বেডিংয়ে রূপান্তর করি, তারপর সমস্ত এম্বেডিংয়ের ওপর কিছু সমষ্টিগত ফাংশন গণনা করি, যেমন `sum`, `average` অথবা `max`।  

![পাঁচটি সিকোয়েন্স শব্দের জন্য একটি এম্বেডিং শ্রেণীবিন্যাসকারী দেখানো ছবি।](../../../../../translated_images/bn/embedding-classifier-example.b77f021a7ee67eee.webp)

আমাদের শ্রেণীবিন্যাসকারী নিউরাল নেটওয়ার্ক শুরু হবে এম্বেডিং লেয়ার দিয়ে, তারপরে অ্যাগগ্রিগেশন লেয়ার, এবং তার ওপর লিনিয়ার শ্রেণীবিন্যাসকারী:


In [2]:
class EmbedClassifier(torch.nn.Module):
    def __init__(self, vocab_size, embed_dim, num_class):
        super().__init__()
        self.embedding = torch.nn.Embedding(vocab_size, embed_dim)
        self.fc = torch.nn.Linear(embed_dim, num_class)

    def forward(self, x):
        x = self.embedding(x)
        x = torch.mean(x,dim=1)
        return self.fc(x)

### ভেরিয়েবল সিকোয়েন্স সাইজ মোকাবেলা করা

এই আর্কিটেকচারের ফলস্বরূপ, আমাদের নেটওয়ার্কের জন্য মিনিব্যাচগুলি একটি নির্দিষ্ট পদ্ধতিতে তৈরি করতে হবে। আগের ইউনিটে, ব্যাগ-অফ-ওয়ার্ডস ব্যবহার করার সময়, একটি মিনিব্যাচে সমস্ত BoW টেনসর সমান সাইজের `vocab_size` ছিল, আমাদের টেক্সট সিকোয়েন্সের প্রকৃত দৈর্ঘ্যের পার্থক্য সত্ত্বেও। একবার আমরা ওয়ার্ড এম্বেডিংসে চলে গেলে, প্রতিটি টেক্সট নমুনায় শব্দের সংখ্যা পরিবর্তনশীল হবে এবং যখন আমরা সেসব নমুনা মিনিব্যাচে একত্রিত করব তখন কিছু প্যাডিং প্রয়োগ করতে হবে।

এটি একই কৌশল ব্যবহার করে করা যায়, যা ডেটাসোর্সে `collate_fn` ফাংশন প্রদান করে:


In [3]:
def padify(b):
    # b is the list of tuples of length batch_size
    #   - first element of a tuple = label, 
    #   - second = feature (text sequence)
    # build vectorized sequence
    v = [encode(x[1]) for x in b]
    # first, compute max length of a sequence in this minibatch
    l = max(map(len,v))
    return ( # tuple of two tensors - labels and features
        torch.LongTensor([t[0]-1 for t in b]),
        torch.stack([torch.nn.functional.pad(torch.tensor(t),(0,l-len(t)),mode='constant',value=0) for t in v])
    )

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=16, collate_fn=padify, shuffle=True)

### এম্বেডিং ক্লাসিফায়ার প্রশিক্ষণ

এখন যেহেতু আমরা সঠিক ডেটালোডার নির্ধারণ করেছি, আমরা আগের ইউনিটে নির্ধারিত প্রশিক্ষণ ফাংশন ব্যবহার করে মডেলটি প্রশিক্ষণ দিতে পারি:


In [4]:
net = EmbedClassifier(vocab_size,32,len(classes)).to(device)
train_epoch(net,train_loader, lr=1, epoch_size=25000)

3200: acc=0.6415625
6400: acc=0.6865625
9600: acc=0.7103125
12800: acc=0.726953125
16000: acc=0.739375
19200: acc=0.75046875
22400: acc=0.7572321428571429


(0.889799795315499, 0.7623160588611644)

> **বিঃদ্রঃ**: আমরা এখানেই মাত্র ২৫ হাজার রেকর্ডের জন্য (পূর্ণ একটি এপোকের কম) সময় বাঁচানোর জন্য প্রশিক্ষণ দিচ্ছি, তবে আপনি প্রশিক্ষণ চালিয়ে যেতে পারেন, একাধিক এপোকের জন্য প্রশিক্ষণের একটি ফাংশন লিখতে পারেন, এবং উচ্চতর সঠিকতা অর্জনের জন্য লার্নিং রেট প্যারামিটারে পরীক্ষা-নিরীক্ষা করতে পারেন। আপনি আনুমানিক ৯০% এর সঠিকতায় পৌঁছাতে সক্ষম হবেন।  


### এম্বেডিংব্যাগ স্তর এবং পরিবর্তনশীল-দৈর্ঘ্যের সিকোয়েন্স উপস্থাপনা

পূর্ববর্তী স্থাপত্যে, আমাদের সব সিকোয়েন্সকে একই দৈর্ঘ্যে প্যাড করতে হয়েছিল যাতে সেগুলো একটি মিনিব্যাচে ফিট করতে পারে। পরিবর্তনশীল দৈর্ঘ্যের সিকোয়েন্স উপস্থাপনের জন্য এটি সবচেয়ে কার্যকর পদ্ধতি নয় - অন্য একটি পন্থা হলো **অফসেট** ভেক্টর ব্যবহার করা, যা একটি বড় ভেক্টরে সংরক্ষিত সমস্ত সিকোয়েন্সের অফসেট ধারণ করবে।

![অফসেট সিকোয়েন্স উপস্থাপনা দেখানো ছবি](../../../../../translated_images/bn/offset-sequence-representation.eb73fcefb29b46ee.webp)

> **নোট**: উপরোক্ত ছবিতে আমরা একটি অক্ষরের সিকোয়েন্স দেখিয়েছি, কিন্তু আমাদের উদাহরণে আমরা শব্দের সিকোয়েন্সের সাথে কাজ করছি। তবে, সিকোয়েন্সগুলোকে অফসেট ভেক্টর দিয়ে উপস্থাপনের সাধারণ নীতি একই থাকে।

অফসেট উপস্থাপনার সাথে কাজ করতে, আমরা [`EmbeddingBag`](https://pytorch.org/docs/stable/generated/torch.nn.EmbeddingBag.html) স্তর ব্যবহার করি। এটি `Embedding` এর মতো, কিন্তু এটি ইনপুট হিসাবে কনটেন্ট ভেক্টর এবং অফসেট ভেক্টর নেয়, এবং এতে একটি গড় করার স্তরও থাকে, যা হতে পারে `mean`, `sum` অথবা `max`।

এখানে পরিবর্তিত নেটওয়ার্ক যা `EmbeddingBag` ব্যবহার করে দেওয়া হলো:


In [5]:
class EmbedClassifier(torch.nn.Module):
    def __init__(self, vocab_size, embed_dim, num_class):
        super().__init__()
        self.embedding = torch.nn.EmbeddingBag(vocab_size, embed_dim)
        self.fc = torch.nn.Linear(embed_dim, num_class)

    def forward(self, text, off):
        x = self.embedding(text, off)
        return self.fc(x)

প্রশিক্ষণের জন্য ডেটাসেট প্রস্তুত করতে, আমাদের একটি রূপান্তর ফাংশন প্রদান করতে হবে যা অফসেট ভেক্টর প্রস্তুত করবে:


In [6]:
def offsetify(b):
    # first, compute data tensor from all sequences
    x = [torch.tensor(encode(t[1])) for t in b]
    # now, compute the offsets by accumulating the tensor of sequence lengths
    o = [0] + [len(t) for t in x]
    o = torch.tensor(o[:-1]).cumsum(dim=0)
    return ( 
        torch.LongTensor([t[0]-1 for t in b]), # labels
        torch.cat(x), # text 
        o
    )

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=16, collate_fn=offsetify, shuffle=True)

মনে রাখবেন, পূর্বের সমস্ত উদাহরণের বিপরীতে, আমাদের নেটওয়ার্ক এখন দুটি প্যারামিটার গ্রহণ করে: ডেটা ভেক্টর এবং অফসেট ভেক্টর, যেগুলোর আকার আলাদা। একইভাবে, আমাদের ডেটা লোডারও আমাদের ২টির পরিবর্তে ৩টি মান প্রদান করে: উভয় টেক্সট এবং অফসেট ভেক্টর ফিচার হিসেবে প্রদান করা হয়। অতএব, আমাদের প্রশিক্ষণ ফাংশনটি সামান্য পরিবর্তন করতে হবে যাতে এটি তা সঠিকভাবে পরিচালনা করতে পারে:


In [7]:
net = EmbedClassifier(vocab_size,32,len(classes)).to(device)

def train_epoch_emb(net,dataloader,lr=0.01,optimizer=None,loss_fn = torch.nn.CrossEntropyLoss(),epoch_size=None, report_freq=200):
    optimizer = optimizer or torch.optim.Adam(net.parameters(),lr=lr)
    loss_fn = loss_fn.to(device)
    net.train()
    total_loss,acc,count,i = 0,0,0,0
    for labels,text,off in dataloader:
        optimizer.zero_grad()
        labels,text,off = labels.to(device), text.to(device), off.to(device)
        out = net(text, off)
        loss = loss_fn(out,labels) #cross_entropy(out,labels)
        loss.backward()
        optimizer.step()
        total_loss+=loss
        _,predicted = torch.max(out,1)
        acc+=(predicted==labels).sum()
        count+=len(labels)
        i+=1
        if i%report_freq==0:
            print(f"{count}: acc={acc.item()/count}")
        if epoch_size and count>epoch_size:
            break
    return total_loss.item()/count, acc.item()/count


train_epoch_emb(net,train_loader, lr=4, epoch_size=25000)

3200: acc=0.6153125
6400: acc=0.6615625
9600: acc=0.6932291666666667
12800: acc=0.715078125
16000: acc=0.7270625
19200: acc=0.7382291666666667
22400: acc=0.7486160714285715


(22.771553103007037, 0.7551983365323096)


## সেম্যান্টিক এমবেডিং: Word2Vec

আমাদের আগের উদাহরণে, মডেল এমবেডিং লেয়ার শব্দকে ভেক্টর প্রতিনিধিত্বে ম্যাপ করতে শিখেছিল, তবে এই প্রতিনিধিত্বের অনেক বেশি সেম্যান্টিক অর্থ ছিল না। এমন একটি ভেক্টর প্রতিনিধিত্ব শিখতে ভালো হবে, যেখানে সাদৃশ্যপূর্ণ শব্দ বা সমার্থক শব্দগুলি এমন ভেক্টরগুলোকে নির্দেশ করবে যা কিছু ভেক্টর দূরত্ব (যেমন ইউক্লিডিয়ান দূরত্ব) অনুযায়ী একে অপরের খুব কাছাকাছি থাকবে।

তা করার জন্য, আমাদের একটি নির্দিষ্ট উপায়ে বড় পরিসরের টেক্সটের উপর আমাদের এমবেডিং মডেলকে প্রি-ট্রেন করতে হবে। সেম্যান্টিক এমবেডিং ট্রেন করার প্রথম পদ্ধতিগুলোর মধ্যে একটি হল [Word2Vec](https://en.wikipedia.org/wiki/Word2vec)। এটি দুটি প্রধান আর্কিটেকচারের উপর ভিত্তি করে যা শব্দগুলোর একটি বিতরণকৃত প্রতিনিধিত্ব তৈরিতে ব্যবহৃত হয়:

 - **কন্টিনিউয়াস ব্যাগ-অফ-ওয়ার্ডস** (CBoW) — এই আর্কিটেকচারে, আমরা মডেলকে প্রদক্ষিণ প্রসঙ্গ থেকে একটি শব্দ পূর্বাভাস দিতে শিখাই। দেওয়া আছে n-gram $(W_{-2},W_{-1},W_0,W_1,W_2)$, মডেলের লক্ষ্য হল $(W_{-2},W_{-1},W_1,W_2)$ থেকে $W_0$ পূর্বাভাস করা।
 - **কন্টিনিউয়াস স্কিপ-গ্রাম** CBoW-এর বিপরীত। মডেল বর্তমান শব্দটি পূর্বাভাস দিতে চারপাশের প্রসঙ্গ শব্দগুলোর উইন্ডো ব্যবহার করে।

CBoW দ্রুত, যেখানে স্কিপ-গ্রাম ধীর, তবে দুর্লভ শব্দের প্রতিনিধিত্বে উন্নত কাজ করে।

![উভয় CBoW এবং Skip-Gram অ্যালগরিদমগুলোকে প্রদর্শন করছে যেগুলো শব্দকে ভেক্টরে পরিণত করে।](../../../../../translated_images/bn/example-algorithms-for-converting-words-to-vectors.fbe9207a726922f6.webp)

Google News ডেটাসেটে প্রি-ট্রেইনড word2vec এমবেডিং নিয়ে পরীক্ষা করার জন্য, আমরা **gensim** লাইব্রেরি ব্যবহার করতে পারি। নিচে আমরা 'neural' শব্দটির সবচেয়ে সাদৃশ্যপূর্ণ শব্দগুলো দেখি

> **নোট:** যখন আপনি প্রথমবার শব্দ ভেক্টর তৈরি করবেন, সেগুলো ডাউনলোড করতে কিছু সময় লাগতে পারে!


In [8]:
import gensim.downloader as api
w2v = api.load('word2vec-google-news-300')

In [9]:
for w,p in w2v.most_similar('neural'):
    print(f"{w} -> {p}")

neuronal -> 0.7804799675941467
neurons -> 0.7326500415802002
neural_circuits -> 0.7252851724624634
neuron -> 0.7174385190010071
cortical -> 0.6941086649894714
brain_circuitry -> 0.6923246383666992
synaptic -> 0.6699118614196777
neural_circuitry -> 0.6638563275337219
neurochemical -> 0.6555314064025879
neuronal_activity -> 0.6531826257705688


আমরা শব্দ থেকে ভেক্টর এম্বেডিংও হিসাব করতে পারি, যা শ্রেণীবিভাগ মডেল প্রশিক্ষণে ব্যবহার করা হবে (পরিষ্কারতার জন্য আমরা শুধুমাত্র ভেক্টরের প্রথম ২০টি উপাদান দেখাচ্ছি):


In [10]:
w2v.word_vec('play')[:20]

array([ 0.01226807,  0.06225586,  0.10693359,  0.05810547,  0.23828125,
        0.03686523,  0.05151367, -0.20703125,  0.01989746,  0.10058594,
       -0.03759766, -0.1015625 , -0.15820312, -0.08105469, -0.0390625 ,
       -0.05053711,  0.16015625,  0.2578125 ,  0.10058594, -0.25976562],
      dtype=float32)

সেমান্তিক এম্বেডিংসের ভালো জিনিস হলো আপনি ভেক্টর এনকোডিং পরিবর্তন করে অর্থ পরিবর্তন করতে পারেন। উদাহরণস্বরূপ, আমরা এমন একটি শব্দ খুঁজে পেতে বলতে পারি, যার ভেক্টর প্রতিনিধিত্ব *king* এবং *woman* শব্দগুলোর কাছাকাছি হবে, এবং *man* শব্দ থেকে যতটা সম্ভব দূরে থাকবে:


In [10]:
w2v.most_similar(positive=['king','woman'],negative=['man'])[0]

('queen', 0.7118192911148071)

উভয় CBoW এবং Skip-Grams হল “পূর্বাভাসমূলক” এমবেডিং, কারণ তারা শুধুমাত্র স্থানীয় প্রসঙ্গগুলি বিবেচনায় নেয়। Word2Vec গ্লোবাল প্রসঙ্গের সুবিধা গ্রহণ করে না। 

**FastText**, Word2Vec এর ওপর নির্মিত যা প্রতিটি শব্দ এবং প্রতিটি শব্দের মধ্যে থাকা ক্যারেক্টার n-গ্রামগুলোর ভেক্টর প্রতিনিধিত্ব শেখে। প্রতিনিধিত্বের মানগুলি তারপর প্রতিটি প্রশিক্ষণ ধাপে একটি ভেক্টরে গড় করা হয়। যদিও এটি প্রি-ট্রেনিং-এ অনেক অতিরিক্ত গণনা যোগ করে, এটি শব্দ এমবেডিংকে সাব-ওয়ার্ড তথ্য এনকোড করতে সক্ষম করে। 

আরেকটি পদ্ধতি, **GloVe**, সহাবস্থান ম্যাট্রিক্স এর ধারণা ব্যবহার করে, নিউরাল পদ্ধতি ব্যবহার করে সহাবস্থান ম্যাট্রিক্সকে আরও প্রকাশ্য এবং নন-লিনিয়ার শব্দ ভেক্টরে ভাঙ্গে।

আপনি উদাহরণের সাথে FastText এবং GloVe-এ এমবেডিং পরিবর্তন করে খেলতে পারেন, কারণ gensim বিভিন্ন শব্দ এমবেডিং মডেল সমর্থন করে।


## PyTorch-এ প্রি-ট্রেইনড এম্বেডিং ব্যবহার করা

উদাহরণটি সংশোধন করে আমরা আমাদের এম্বেডিং লেয়ারের ম্যাট্রিক্সটি সেমান্টিক এম্বেডিং যেমন Word2Vec দিয়ে পূর্ব-পরিপূর্ণ করতে পারি। আমাদের খেয়াল রাখতে হবে যে, প্রি-ট্রেইনড এম্বেডিং এবং আমাদের টেক্সট কর্পাসের শব্দভাণ্ডার সম্ভবত এক নয়, তাই অনুপস্থিত শব্দগুলির জন্য ওজনগুলি র্যান্ডম মান দিয়ে ইনিশিয়ালাইজ করব:


In [11]:
embed_size = len(w2v.get_vector('hello'))
print(f'Embedding size: {embed_size}')

net = EmbedClassifier(vocab_size,embed_size,len(classes))

print('Populating matrix, this will take some time...',end='')
found, not_found = 0,0
for i,w in enumerate(vocab.get_itos()):
    try:
        net.embedding.weight[i].data = torch.tensor(w2v.get_vector(w))
        found+=1
    except:
        net.embedding.weight[i].data = torch.normal(0.0,1.0,(embed_size,))
        not_found+=1

print(f"Done, found {found} words, {not_found} words missing")
net = net.to(device)

Embedding size: 300
Populating matrix, this will take some time...Done, found 41080 words, 54732 words missing


এখন আমাদের মডেলটি প্রশিক্ষণ দিই। লক্ষ করুন, মডেল প্রশিক্ষণের সময় আগের উদাহরণের তুলনায় উল্লেখযোগ্যভাবে বেশি কারণ এমবেডিং স্তরের আকার বড় এবং সেই অনুযায়ী অনেক বেশি প্যারামিটার রয়েছে। এছাড়াও, এর কারণে, ওভারফিটিং এড়ানোর জন্য আমাদের মডেলটি আরও উদাহরণে প্রশিক্ষণ দেওয়ার প্রয়োজন হতে পারে।


In [12]:
train_epoch_emb(net,train_loader, lr=4, epoch_size=25000)

3200: acc=0.6359375
6400: acc=0.68109375
9600: acc=0.7067708333333333
12800: acc=0.723671875
16000: acc=0.73625
19200: acc=0.7463541666666667
22400: acc=0.7560714285714286


(214.1013875559821, 0.7626759436980166)

আমাদের ক্ষেত্রে আমরা সঠিকতার বিশাল বৃদ্ধি দেখতে পাইনি, যা সম্ভবত বিভিন্ন শব্দভান্ডারের কারণে। 
বিভিন্ন শব্দভান্ডারের সমস্যাটি অতিক্রান্ত করতে, আমরা নিম্নলিখিত সমাধানগুলির একটি ব্যবহার করতে পারি:
* word2vec মডেলকে আমাদের শব্দভান্ডারে পুনরায় প্রশিক্ষণ দিন
* পূর্বপ্রশিক্ষিত word2vec মডেলের শব্দভান্ডার দিয়ে আমাদের ডেটাসেট লোড করুন। ডেটাসেট লোড করার সময় ব্যবহৃত শব্দভান্ডার নির্দিষ্ট করা যেতে পারে।

পরের পদ্ধতিটি সহজ মনে হয়, বিশেষ করে কারণ PyTorch `torchtext` ফ্রেমওয়ার্ক এম্বেডিংগুলির জন্য বিল্ট-ইন সাপোর্ট নিয়ে আসে। উদাহরণস্বরূপ, আমরা নিম্নলিখিত প্রণালীতে GloVe-ভিত্তিক শব্দভান্ডার উদ্ঘাটন করতে পারি:


In [14]:
vocab = torchtext.vocab.GloVe(name='6B', dim=50)

100%|█████████▉| 399999/400000 [00:15<00:00, 25411.14it/s]


Loaded vocabulary has the following basic operations:
* `vocab.stoi` dictionary allows us to convert word into its dictionary index
* `vocab.itos` does the opposite - converts number into word
* `vocab.vectors` is the array of embedding vectors, so to get the embedding of a word `s` we need to use `vocab.vectors[vocab.stoi[s]]`

Here is the example of manipulating embeddings to demonstrate the equation **kind-man+woman = queen** (I had to tweak the coefficient a bit to make it work):


In [15]:
# get the vector corresponding to kind-man+woman
qvec = vocab.vectors[vocab.stoi['king']]-vocab.vectors[vocab.stoi['man']]+1.3*vocab.vectors[vocab.stoi['woman']]
# find the index of the closest embedding vector 
d = torch.sum((vocab.vectors-qvec)**2,dim=1)
min_idx = torch.argmin(d)
# find the corresponding word
vocab.itos[min_idx]

'queen'

ঐ এমবেডিংগুলি ব্যবহার করে শ্রেণীবিন্যাসকারী প্রশিক্ষণ দেওয়ার জন্য, আমাদের প্রথমে GloVe শব্দভাণ্ডার ব্যবহার করে আমাদের ডেটাসেট এনকোড করতে হবে:


In [16]:
def offsetify(b):
    # first, compute data tensor from all sequences
    x = [torch.tensor(encode(t[1],voc=vocab)) for t in b] # pass the instance of vocab to encode function!
    # now, compute the offsets by accumulating the tensor of sequence lengths
    o = [0] + [len(t) for t in x]
    o = torch.tensor(o[:-1]).cumsum(dim=0)
    return ( 
        torch.LongTensor([t[0]-1 for t in b]), # labels
        torch.cat(x), # text 
        o
    )

উপরে আমরা দেখেছি, সমস্ত ভেক্টর এম্বেডিংগুলি `vocab.vectors` ম্যাট্রিক্সে সংরক্ষিত হয়। এটি এম্বেডিং স্তরের ওজনগুলিতে ঐ ওজনগুলি פשוט কপি করে লোড করা অত্যন্ত সহজ করে তোলে:


In [17]:
net = EmbedClassifier(len(vocab),len(vocab.vectors[0]),len(classes))
net.embedding.weight.data = vocab.vectors
net = net.to(device)

এখন আসুন আমাদের মডেলটি প্রশিক্ষণ করি এবং দেখি আমরা কি ভাল ফলাফল পাচ্ছি কি না:


In [18]:
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=16, collate_fn=offsetify, shuffle=True)
train_epoch_emb(net,train_loader, lr=4, epoch_size=25000)

3200: acc=0.6271875
6400: acc=0.68078125
9600: acc=0.7030208333333333
12800: acc=0.71984375
16000: acc=0.7346875
19200: acc=0.7455729166666667
22400: acc=0.7529464285714286


(35.53972978646833, 0.7575175943698017)

সঠিকতার উল্লেখযোগ্য বাড়তি দেখা না যাওয়ার একটি কারণ হল যে আমাদের ডেটাসেটের কিছু শব্দ প্রাক-প্রশিক্ষিত GloVe শব্দভান্ডারে অনুপস্থিত, এবং তাই সেগুলি মৌলিকভাবে অগ্রাহ্য করা হয়। এই পরিস্থিতি কাটিয়ে উঠতে আমরা আমাদের ডেটাসেটে নিজস্ব এম্বেডিং ট্রেন করতে পারি। 


## প্রসঙ্গভিত্তিক এম্বেডিংস

প্রথাগত প্রি-ট্রেইনড এম্বেডিং প্রতিনিধিত্ব যেমন Word2Vec এর একটি মূল সীমাবদ্ধতা হল শব্দের অর্থ নির্ধারণের সমস্যা। যদিও প্রি-ট্রেইনড এম্বেডিংস প্রসঙ্গে শব্দগুলির কিছু অর্থ ধারণ করতে পারে, একটি শব্দের প্রতিটি সম্ভাব্য অর্থ একই এম্বেডিংয়ে এনকোড করা হয়। এটি ডাউনস্ট্রীম মডেলগুলিতে সমস্যা সৃষ্টি করতে পারে, কারণ অনেক শব্দ যেমন 'play' শব্দটির বিভিন্ন অর্থ থাকে ব্যবহৃত প্রসঙ্গ অনুযায়ী।

উদাহরণস্বরূপ, 'play' শব্দটির নিচের দুইটি বাক্যে যথেষ্ট ভিন্ন অর্থ রয়েছে:
- আমি থিয়েটারে একটি **play** এ গিয়েছিলাম।
- জন তার বন্ধুদের সাথে **play** করতে চায়।

উপরের প্রি-ট্রেইনড এম্বেডিংস 'play' শব্দটির উভয় অর্থ একই এম্বেডিংয়ে উপস্থাপন করে। এই সীমাবদ্ধতা কাটিয়ে উঠতে, আমাদের **ভাষা মডেল** এর উপর ভিত্তি করে এম্বেডিং তৈরি করতে হবে, যা একটি বড় টেক্সট কর্পাসে প্রশিক্ষিত, এবং *জানে* কীভাবে শব্দগুলো ভিন্ন প্রসঙ্গে একত্রে স্থাপন করা যায়। প্রসঙ্গভিত্তিক এম্বেডিংস নিয়ে আলোচনা এই টিউটোরিয়ালের আওতার বাইরে, তবে পরবর্তী ইউনিটে ভাষা মডেল সম্পর্কে আলোচনা করার সময় আমরা আবার এগুলো নিয়ে আসব।


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**অস্বীকৃতি**:
এই নথিটি AI অনুবাদ পরিষেবা [Co-op Translator](https://github.com/Azure/co-op-translator) ব্যবহার করে অনূদিত হয়েছে। যদিও আমরা শুদ্ধতার জন্য চেষ্টা করি, অনুগ্রহ করে মনে রাখবেন যে স্বয়ংক্রিয় অনুবাদে ত্রুটি বা অসঙ্গতি থাকতে পারে। মূল নথিটি তার স্বভাষায় কর্তৃত্বপূর্ণ উৎস হিসেবে বিবেচিত হওয়া উচিত। গুরুত্বপূর্ণ তথ্যের জন্য পেশাদার মানব অনুবাদ সুপারিশ করা হয়। এই অনুবাদের ব্যবহারে প্রয়োজনীয় ভুল বোঝাবুঝি বা ভুল ব্যাখ্যার জন্য আমরা দায়বদ্ধ নই।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
